In [67]:
import pandas as pd

In [68]:
df_dim_region   = pd.read_parquet('../../data/silver/dim_region.parquet')
df_dim_product  = pd.read_parquet('../../data/silver/dim_product.parquet')
df_dim_calendar = pd.read_parquet('../../data/silver/dim_calendar.parquet')
df_dim_supplier = pd.read_parquet('../../data/silver/dim_supplier.parquet')
df_fact_sales   = pd.read_parquet('../../data/silver/fact_sales_weekly.parquet')

In [69]:
df_dim_region.to_parquet('../../data/gold/data_warehouse/dw_dim_region.parquet', index=False)
df_dim_product.to_parquet('../../data/gold/data_warehouse/dw_dim_product.parquet', index=False)
df_dim_supplier.to_parquet('../../data/gold/data_warehouse/dw_dim_supplier.parquet', index=False)
df_fact_sales.to_parquet('../../data/gold/data_warehouse/dw_fact_sales_weekly.parquet', index=False)

In [70]:
columns = df_dim_calendar.columns[1:]
df_dim_weekly_calendar = (
    df_dim_calendar
    .groupby('week_date', as_index=False)
    .agg(
        start_date=('date', 'min'),
        end_date=('date', 'max'),
        **{col: (col, 'first') for col in columns}
    )
    .drop(columns=[col for col in columns if 'day' in col])
)
weekly_columns = ['week_date','week']
for col in df_dim_weekly_calendar.columns[:-2]:
    weekly_columns.append(col)

df_dim_weekly_calendar = df_dim_weekly_calendar[weekly_columns]
df_dim_weekly_calendar.head(5)

,week_date,week,start_date,end_date,year,semester,semester_date,semester_name,quarter,quarter_date,quarter_name,month,month_name,month_date
0,2022-10-31,44,2022-10-31,2022-11-06,2022,2,2022-10-01,H2,4,2022-10-01,Q4,10,October,2022-10-01
1,2022-11-07,45,2022-11-07,2022-11-13,2022,2,2022-11-01,H2,4,2022-10-01,Q4,11,November,2022-11-01
2,2022-11-14,46,2022-11-14,2022-11-20,2022,2,2022-11-01,H2,4,2022-10-01,Q4,11,November,2022-11-01
3,2022-11-21,47,2022-11-21,2022-11-27,2022,2,2022-11-01,H2,4,2022-10-01,Q4,11,November,2022-11-01
4,2022-11-28,48,2022-11-28,2022-12-04,2022,2,2022-11-01,H2,4,2022-10-01,Q4,11,November,2022-11-01


In [71]:
df_dim_weekly_calendar.to_parquet('../../data/gold/data_warehouse/dw_dim_weekly_calendar.parquet', index=False)

In [72]:
len(df_dim_weekly_calendar['week_date'].unique())

104

## Active combinations

Only (supplier, product, region) combos that actually had sales are kept.
This naturally excludes empty time series without any extra filtering step.

In [73]:
df_fact_sales

,sales_id,week_date,supplier_id,region_id,product_id,units_sold
0,1,2022-10-31,1,5,97,2
1,2,2022-10-31,1,5,121,61
2,3,2022-10-31,1,5,134,189
3,4,2022-10-31,1,5,241,5064
4,5,2022-10-31,1,5,246,22
...,...,...,...,...,...,...
128812,128813,2024-10-21,27,4,255,45
128813,128814,2024-10-21,27,4,258,3
128814,128815,2024-10-21,27,4,278,15
128815,128816,2024-10-21,27,4,333,1350


In [74]:
import pandas as pd
import numpy as np

def create_time_series_metadata(df_fact_sales):
    """
    Cria metadados de séries temporais COM spine completo
    
    Parameters:
    -----------
    df_fact_sales : DataFrame
        Tabela de fatos com vendas (pode ter gaps)
    
    Returns:
    --------
    DataFrame : Metadados com estatísticas CORRETAS (incluindo zeros)
    """
    
    print("🔄 Criando spine completo por série temporal...")
    
    results = []
    total_series = df_fact_sales.groupby([
        'supplier_id', 'region_id', 'product_id'
    ]).ngroups
    
    print(f"   Total de séries: {total_series}")
    
    for i, ((supplier, region, product), group) in enumerate(
        df_fact_sales.groupby(['supplier_id', 'region_id', 'product_id'])
    ):
        if (i + 1) % 500 == 0:
            print(f"   Processando: {i+1}/{total_series}")
        
        # Período ativo
        first_week = group['week_date'].min()
        last_week = group['week_date'].max()
        
        # Calendário completo para esse período
        series_weeks = pd.date_range(first_week, last_week, freq='W-MON')
        
        # Spine
        df_spine = pd.DataFrame({
            'supplier_id': supplier,
            'region_id': region,
            'product_id': product,
            'week_date': series_weeks
        })
        
        # Merge com dados reais
        df_series = df_spine.merge(
            group[['week_date', 'units_sold']],
            on='week_date',
            how='left'
        )
        
        # Preencher zeros
        df_series['units_sold'] = df_series['units_sold'].fillna(0)
        
        # Calcular estatísticas
        units = df_series['units_sold'].values
        
        result = {
            'supplier_id': supplier,
            'region_id': region,
            'product_id': product,
            'first_week_date': first_week,
            'last_week_date': last_week,
            'total_weeks_length': len(series_weeks),
            'num_week_with_sales': (units > 0).sum(),
            'num_week_with_zeros': (units == 0).sum(),
            'sales_units': units.sum(),
            'avg_weekly_sales': units.mean(),
            'std_weekly_sales': units.std(ddof=1) if len(units) > 1 else 0,
            'max_weekly_sales': units.max(),
            'min_weekly_sales': units.min(),
            'q25_sales': np.quantile(units, 0.25),
            'q50_sales': np.quantile(units, 0.50),  # Mediana
            'q75_sales': np.quantile(units, 0.75)
        }
        
        results.append(result)
    
    df_metadata = pd.DataFrame(results)
    
    print(f"\n✅ Metadados criados: {len(df_metadata)} séries")
    
    # Features derivadas
    df_metadata['sales_weeks_ratio'] = (
        df_metadata['num_week_with_sales'] / df_metadata['total_weeks_length']
    )
    
    df_metadata['cv'] = (
        df_metadata['std_weekly_sales'] / 
        (df_metadata['avg_weekly_sales'] + 1e-6)
    )
    
    df_metadata['iqr'] = df_metadata['q75_sales'] - df_metadata['q25_sales']

    
    return df_metadata

df_dim_time_series = create_time_series_metadata(df_fact_sales)

🔄 Criando spine completo por série temporal...
   Total de séries: 2845
   Processando: 500/2845
   Processando: 1000/2845
   Processando: 1500/2845
   Processando: 2000/2845
   Processando: 2500/2845

✅ Metadados criados: 2845 séries


In [75]:
empty_series_count = (df_dim_time_series['num_week_with_sales'] == 0).sum()
print(f"\n📊 Séries sem vendas: {empty_series_count} ({empty_series_count / len(df_dim_time_series) * 100:.2f}%)")


📊 Séries sem vendas: 4 (0.14%)


In [76]:
df_dim_time_series[df_dim_time_series['num_week_with_sales'] == 0]

,supplier_id,region_id,product_id,first_week_date,last_week_date,total_weeks_length,num_week_with_sales,num_week_with_zeros,sales_units,avg_weekly_sales,std_weekly_sales,max_weekly_sales,min_weekly_sales,q25_sales,q50_sales,q75_sales,sales_weeks_ratio,cv,iqr
90,3,2,66,2022-10-31,2023-12-04,58,0,58,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
124,3,2,143,2022-10-31,2024-03-18,73,0,73,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
405,6,1,143,2023-10-23,2024-02-19,18,0,18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1293,6,4,143,2023-08-21,2024-09-30,59,0,59,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [77]:
#remove empty time series
df_dim_time_series = df_dim_time_series[df_dim_time_series['num_week_with_sales'] > 0]

In [78]:
df_dim_time_series.reset_index().rename(columns={'index': 'time_series_id'}, inplace=True)
df_dim_time_series['time_series_id'] = df_dim_time_series.index + 1
df_dim_time_series = df_dim_time_series[[ 'time_series_id' ] + [col for col in df_dim_time_series.columns[:-1]]]
df_dim_time_series.to_parquet('../../data/gold/data_warehouse/dw_dim_time_series.parquet', index=False)
df_dim_time_series

,time_series_id,supplier_id,region_id,product_id,first_week_date,last_week_date,total_weeks_length,num_week_with_sales,num_week_with_zeros,sales_units,avg_weekly_sales,std_weekly_sales,max_weekly_sales,min_weekly_sales,q25_sales,q50_sales,q75_sales,sales_weeks_ratio,cv,iqr
0,1,1,5,69,2023-01-09,2023-08-21,33,4,29,10.0,0.303030,1.103541,6.0,0.0,0.00,0.0,0.00,0.121212,3.641674,0.0
1,2,1,5,89,2023-03-06,2024-10-21,86,33,53,1716.0,19.953488,65.791275,375.0,0.0,0.00,0.0,6.00,0.383721,3.297232,6.0
2,3,1,5,97,2022-10-31,2023-01-30,14,3,11,5.0,0.357143,0.744946,2.0,0.0,0.00,0.0,0.00,0.214286,2.085844,0.0
3,4,1,5,121,2022-10-31,2024-10-21,104,83,21,9966.0,95.826923,90.341149,393.0,0.0,17.25,82.0,145.25,0.798077,0.942753,128.0
4,5,1,5,134,2022-10-31,2024-10-21,104,48,56,14208.0,136.615385,670.247126,5646.0,0.0,0.00,0.0,39.00,0.461538,4.906088,39.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2840,2841,27,4,351,2022-11-28,2023-11-20,52,5,47,17.0,0.326923,1.688730,12.0,0.0,0.00,0.0,0.00,0.096154,5.165512,0.0
2841,2842,27,4,357,2022-11-07,2024-10-07,101,26,75,58.0,0.574257,1.321715,7.0,0.0,0.00,0.0,1.00,0.257426,2.301603,1.0
2842,2843,27,4,373,2022-12-05,2024-10-21,99,44,55,178.0,1.797980,3.401620,19.0,0.0,0.00,0.0,2.00,0.444444,1.891911,2.0
2843,2844,27,4,377,2022-10-31,2024-09-30,101,67,34,263.0,2.603960,2.946453,13.0,0.0,0.00,2.0,4.00,0.663366,1.131527,4.0


In [79]:
(
    df_fact_sales
    .merge(df_dim_time_series, on=['supplier_id', 'region_id', 'product_id'], how='inner')
    [['week_date','supplier_id', 'region_id', 'product_id','time_series_id','units_sold']]
    .to_parquet('../../data/gold/data_warehouse/dw_fact_sales_weekly.parquet', index=False
    )
)

In [80]:
df_fact_sales

,sales_id,week_date,supplier_id,region_id,product_id,units_sold
0,1,2022-10-31,1,5,97,2
1,2,2022-10-31,1,5,121,61
2,3,2022-10-31,1,5,134,189
3,4,2022-10-31,1,5,241,5064
4,5,2022-10-31,1,5,246,22
...,...,...,...,...,...,...
128812,128813,2024-10-21,27,4,255,45
128813,128814,2024-10-21,27,4,258,3
128814,128815,2024-10-21,27,4,278,15
128815,128816,2024-10-21,27,4,333,1350
